# In Class Activity April 14th, 2026

In [1]:
pip install optuna


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# importing data
adult = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homwork-3/In Class Activities/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





There is lots of skewness in the data meaning there could be class imbalances. Lots of adults appear to be apart of the Private working class, most are High school graduates, their maritial status have a majority of married-civ-spouse. 

- Age is skewed to the right with most adults between 25-50 years old and fewer older individuals, which is typical of a working-age population.
- Workclass shows that most individuals in the dataset are privately employed while others are less frequent.
- Education showed that High School grads and some college are the most common categories and other degrees like Masters are less frequent.

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace(r'^\s*\?\s*$', np.nan, regex=True)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1, errors='ignore')

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if str(x).strip() == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

With a mean F1 Score of 0.71196, the baseline XGBoost model managed the categorical variables without needing extensive one-hot encoding, but there can be room to improve it by tuning RandomizedSearchCV and Optuna. This means using scale_pos_weight, since income levels are imbalanced adjusting this weight will help the model pay more attention to lower income levels of >50k. learning_rate is also important and by tuning this I can help the sequential boosting process with their optimal weights. 

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [8]:
# your code here
# max_depth: Turned max_depth down to 3 and making sure it less prone to overfitting
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_depth = XGBClassifier(enable_categorical=True, random_state=42, max_depth=3)
depth_scores = cross_val_score(xgb_depth, X, y, cv=skf, scoring='f1')
print(f'1. Max Depth = 3 | Mean F1: {depth_scores.mean():.4f}')

# learning_rate: lower the learning_rate to 0.05 so the model takes time learning the steps 
xgb_lr = XGBClassifier(enable_categorical=True, random_state=42, learning_rate=0.05)
lr_scores = cross_val_score(xgb_lr, X, y, cv=skf, scoring='f1')
print(f'2. Learning Rate = 0.05 | Mean F1: {lr_scores.mean():.4f}')

# scale_pos_weight: set it to 3 to make sure it gets it right 
xgb_scale = XGBClassifier(enable_categorical=True, random_state=42, scale_pos_weight=3.0)
scale_scores = cross_val_score(xgb_scale, X, y, cv=skf, scoring='f1')
print(f'3. Scale Pos Weight = 3.0 | Mean F1: {scale_scores.mean():.4f}')


1. Max Depth = 3 | Mean F1: 0.7122
2. Learning Rate = 0.05 | Mean F1: 0.6959
3. Scale Pos Weight = 3.0 | Mean F1: 0.7166


scale_pos_weight and max_depth features gave a higher CV F1 score 

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [9]:
# your code here
base_xgb = XGBClassifier(enable_categorical=True, random_state=42, scale_pos_weight=3.0)
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [50, 100],
    'min_child_weight': [1, 3]
}
grid_search = GridSearchCV(
    estimator=base_xgb, 
    param_grid=param_grid, 
    cv=skf, 
    scoring='f1', 
    n_jobs=-1)

grid_search.fit(X_train, y_train)
best_grid_model = grid_search.best_estimator_
y_pred = best_grid_model.predict(X_test)
print(f"\nBest Parameters Found: {grid_search.best_params_}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")


Best Parameters Found: {'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 100}
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.88      7431
           1       0.61      0.86      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [10]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    'n_estimators': np.arange(50, 300, 50), 
    'min_child_weight': np.arange(1, 8), 
}

# replace this placeholder model with your preferred model from above
base_xgb_random = XGBClassifier(random_state=42, enable_categorical=True, scale_pos_weight=3.0)

xgb_random = RandomizedSearchCV(base_xgb_random,
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')  

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, 
                                scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                n_estimators=xgb_random.best_params_['n_estimators'],       
                                min_child_weight=xgb_random.best_params_['min_child_weight'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')

Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(3.0), 'n_estimators': np.int64(150), 'min_child_weight': np.int64(6), 'max_depth': np.int64(10), 'learning_rate': np.float64(0.07444444444444444)}
Best F1 score from RandomizedSearchCV: 0.7161853763707312
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7431
           1       0.63      0.84      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.79      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [12]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)     # <-- Added
    min_child_weight = trial.suggest_int('min_child_weight', 1, 7)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, 
                               scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  
                               learning_rate=learning_rate, 
                               n_estimators=n_estimators,         # <-- Added
                               min_child_weight=min_child_weight, # <-- Added
                               enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  n_estimators=study.best_params['n_estimators'],      
                                  min_child_weight=study.best_params['min_child_weight'],
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 10:03:40,724] A new study created in memory with name: no-name-7512d95f-608d-4066-9a78-139b44983c77
Best trial: 0. Best value: 0.723159:   5%|▌         | 1/20 [00:05<01:40,  5.27s/it]

[I 2026-04-15 10:03:45,990] Trial 0 finished with value: 0.723159291427534 and parameters: {'scale_pos_weight': 2.243108158781344, 'max_depth': 9, 'learning_rate': 0.07718831103002484, 'n_estimators': 113, 'min_child_weight': 1}. Best is trial 0 with value: 0.723159291427534.


Best trial: 0. Best value: 0.723159:  10%|█         | 2/20 [00:10<01:33,  5.19s/it]

[I 2026-04-15 10:03:51,140] Trial 1 finished with value: 0.6976140127969978 and parameters: {'scale_pos_weight': 7.752274016128518, 'max_depth': 8, 'learning_rate': 0.2869754389167913, 'n_estimators': 213, 'min_child_weight': 1}. Best is trial 0 with value: 0.723159291427534.


Best trial: 2. Best value: 0.724334:  15%|█▌        | 3/20 [00:12<01:04,  3.78s/it]

...

[I 2026-04-15 10:04:37,424] Trial 17 finished with value: 0.6562205471764747 and parameters: {'scale_pos_weight': 6.687729834744648, 'max_depth': 9, 'learning_rate': 0.014050747856529378, 'n_estimators': 158, 'min_child_weight': 5}. Best is trial 2 with value: 0.7243335842241334.


Best trial: 2. Best value: 0.724334:  95%|█████████▌| 19/20 [00:58<00:02,  2.65s/it]

[I 2026-04-15 10:04:39,627] Trial 18 finished with value: 0.6943098920950508 and parameters: {'scale_pos_weight': 4.528989665919182, 'max_depth': 7, 'learning_rate': 0.06107630461935226, 'n_estimators': 106, 'min_child_weight': 1}. Best is trial 2 with value: 0.7243335842241334.


Best trial: 19. Best value: 0.729035: 100%|██████████| 20/20 [01:00<00:00,  3.04s/it]


[I 2026-04-15 10:04:41,441] Trial 19 finished with value: 0.7290349728453313 and parameters: {'scale_pos_weight': 1.8858592641558172, 'max_depth': 8, 'learning_rate': 0.1640917981875743, 'n_estimators': 85, 'min_child_weight': 4}. Best is trial 19 with value: 0.7290349728453313.
Best parameters from Optuna: {'scale_pos_weight': 1.8858592641558172, 'max_depth': 8, 'learning_rate': 0.1640917981875743, 'n_estimators': 85, 'min_child_weight': 4}
Best F1 score from Optuna: 0.7290349728453313
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      7431
           1       0.69      0.77      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.81      0.83      0.82      9769
weighted avg       0.87      0.86      0.87      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?

GridSearchCV: This method was more slow to run on my model because it kept trying every single combination of parameters. Because of this I had to keep my parameter options small to avoid the model from taking too long to train.
RandomizedSearchCV: This process felt more efficient as I only tested 20 random combinations, I was able to test larger ranges of numbers the n_estimators up to 300 without crashing my computer. Because it relies on randomness, it doesn't "learn" from its bad guesses. 
Optuna: This was the most advanced approach because it evaluated its past trials and learns where the good parameters are making it smarter with guesses over time rather than just randomly making guesses.

Best Results and prefernces: 
For the actual metrics, Optuna gave the best overall F1 score of 0.73. Optuna not only does its learning outperform random guessing gien enough trials, the code structure was what I found to be much cleaner.